# 自分のチャットボットを作る (LoRA で追加学習)

日本語をすでに話せる公開モデル (Qwen2.5-Instruct) に、自分の「質問と答え」を追加で覚えさせます。
一から作るちいさなGPTと違って、教えていない質問にもそれらしく答えられます。そこに自分の知識・口調・キャラクターを足すのが目的です。

1. メニューの **ランタイム → ランタイムのタイプを変更** で **T4 GPU** を選ぶ
2. 上のセルから順に実行する (▶ ボタン、または Shift+Enter)
3. 途中のセルで、ブラウザから話しかけられるチャット画面の URL が出ます (Colab を開いている間だけ)
4. 最後の「ブラウザで動く形にして公開する」まで進むと、Colab を閉じても使える ChatGPT のようなページが Hugging Face にできます

In [ ]:
!pip install -q "transformers>=4.56" peft accelerate
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "なし (ランタイムのタイプを T4 GPU にしてください)")

## 1. 追加学習のコード

In [ ]:
%%writefile chat_finetune.py
"""公開されている小さなチャットモデルを、自分の「質問と答え」で追加学習 (LoRA) する。

ちいさなGPT工房の「質問: … / 答え: …」と同じ形の文章をそのまま使える。
一から作るモデルと違い、もとのモデルが日本語や一般常識をすでに知っているので、
教えていない質問にもそれらしく答えられる。そこに自分の知識や口調を足すのが LoRA。

    python chat_finetune.py my_qa.txt --model Qwen/Qwen2.5-1.5B-Instruct
"""

import argparse
import math
import random
import re

import torch

DEFAULT_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
DEFAULT_SYSTEM = "あなたは親切なアシスタントです。日本語で簡潔に答えます。"


# ---------- データ ----------
def parse_qa(text):
    """「質問: …」「答え: …」の組を空行で区切った文章を (質問, 答え) のリストにする。
    答えは次の空行まで何行でもよい。「Q:」「A:」でも読める。"""
    pairs = []
    for block in re.split(r"\n\s*\n", text.strip()):
        m = re.match(r"\s*(?:質問|Q)\s*[:：]\s*(.+?)\n\s*(?:答え|A)\s*[:：]\s*(.+)", block, re.S)
        if m:
            pairs.append((m.group(1).strip(), m.group(2).strip()))
    return pairs


def _chat_text(tok, messages, add_generation_prompt):
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=add_generation_prompt)


def build_examples(tok, pairs, system=DEFAULT_SYSTEM, max_len=512):
    """答えの部分だけを学習するよう、質問側のラベルを -100 にする。"""
    examples = []
    for q, a in pairs:
        head = [{"role": "system", "content": system}, {"role": "user", "content": q}] if system else \
               [{"role": "user", "content": q}]
        prompt = _chat_text(tok, head, True)
        full = _chat_text(tok, head + [{"role": "assistant", "content": a}], False)
        if not full.startswith(prompt):
            raise ValueError("このモデルの会話テンプレートには対応していません")
        p_ids = tok(prompt, add_special_tokens=False)["input_ids"]
        f_ids = tok(full, add_special_tokens=False)["input_ids"][:max_len]
        labels = [-100] * min(len(p_ids), len(f_ids)) + f_ids[len(p_ids):]
        examples.append((f_ids, labels))
    return examples


def _batches(examples, batch_size, pad_id, device):
    order = list(range(len(examples)))
    random.shuffle(order)
    for i in range(0, len(order), batch_size):
        chunk = [examples[j] for j in order[i:i + batch_size]]
        L = max(len(ids) for ids, _ in chunk)
        ids = torch.full((len(chunk), L), pad_id, dtype=torch.long)
        lab = torch.full((len(chunk), L), -100, dtype=torch.long)
        att = torch.zeros((len(chunk), L), dtype=torch.long)
        for k, (x, y) in enumerate(chunk):
            ids[k, :len(x)] = torch.tensor(x)
            lab[k, :len(y)] = torch.tensor(y)
            att[k, :len(x)] = 1
        yield ids.to(device), att.to(device), lab.to(device)


# ---------- モデル ----------
def load(model_name=DEFAULT_MODEL, device=None):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    tok = AutoTokenizer.from_pretrained(model_name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    # T4 は bfloat16 が遅いので float16 で読む
    dtype = torch.float16 if device == "cuda" else torch.float32
    model = AutoModelForCausalLM.from_pretrained(model_name, dtype=dtype).to(device)
    return model, tok


def add_lora(model, r=16, alpha=32, dropout=0.05):
    from peft import LoraConfig, get_peft_model
    # 注意と MLP のすべての線形層に、小さな行列 (ランク r) を足して、そこだけ学習する
    cfg = LoraConfig(r=r, lora_alpha=alpha, lora_dropout=dropout, task_type="CAUSAL_LM",
                     target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
    model = get_peft_model(model, cfg)
    model.print_trainable_parameters()
    return model


def train(model, tok, pairs, system=DEFAULT_SYSTEM, epochs=3, batch_size=8, lr=2e-4, log=print):
    device = next(model.parameters()).device
    examples = build_examples(tok, pairs, system)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
    total = epochs * math.ceil(len(examples) / batch_size)
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    log(f"質問と答え {len(examples)} 組 / {epochs} 周 / {total} ステップ")
    model.train()
    step = 0
    for ep in range(epochs):
        losses = []
        for ids, att, lab in _batches(examples, batch_size, tok.pad_token_id, device):
            for g in opt.param_groups:  # コサインで下げる
                g["lr"] = lr * 0.5 * (1 + math.cos(math.pi * step / max(1, total)))
            with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
                loss = model(input_ids=ids, attention_mask=att, labels=lab).loss
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt)
            scaler.update()
            losses.append(loss.item())
            step += 1
        log(f"{ep + 1} 周目  損失 {sum(losses) / len(losses):.3f}")
    model.eval()
    return model


@torch.no_grad()
def chat(model, tok, question, history=(), system=DEFAULT_SYSTEM, max_new_tokens=256, temperature=0.7):
    """history は [(質問, 答え), ...]。"""
    msgs = [{"role": "system", "content": system}] if system else []
    for q, a in history:
        msgs += [{"role": "user", "content": q}, {"role": "assistant", "content": a}]
    msgs.append({"role": "user", "content": question})
    enc = tok(_chat_text(tok, msgs, True), return_tensors="pt", add_special_tokens=False)
    enc = {k: v.to(next(model.parameters()).device) for k, v in enc.items()}
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=temperature > 0,
                         temperature=temperature if temperature > 0 else None, top_p=0.9 if temperature > 0 else None,
                         pad_token_id=tok.pad_token_id)
    return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("qa", help="「質問: …」「答え: …」の文章 (UTF-8)")
    ap.add_argument("--model", default=DEFAULT_MODEL)
    ap.add_argument("--system", default=DEFAULT_SYSTEM)
    ap.add_argument("--epochs", type=int, default=3)
    ap.add_argument("--batch", type=int, default=8)
    ap.add_argument("--lr", type=float, default=2e-4)
    ap.add_argument("--out", default="my_lora")
    a = ap.parse_args()
    with open(a.qa, encoding="utf-8") as f:
        pairs = parse_qa(f.read())
    model, tok = load(a.model)
    model = add_lora(model)
    train(model, tok, pairs, a.system, a.epochs, a.batch, a.lr)
    model.save_pretrained(a.out)
    tok.save_pretrained(a.out)
    print(f"{a.out} に保存しました")
    print(chat(model, tok, pairs[0][0], system=a.system))


if __name__ == "__main__":
    main()

## 2. 質問と答えのデータ

ちいさなGPT工房と同じ形です。「質問: …」「答え: …」を空行で区切って並べます。答えは何行でもかまいません。

```
質問: あなたはだれですか。
答え: わたしは山の郵便局で育った、小さな言語モデルです。

質問: 局長はだれですか。
答え: 局長は、ねこです。
```

まずは例のデータを書き出します。自分のデータを使うときは、その次のセルでアップロードしてください。
目安は 50〜数千組です。同じことを聞く言い方を何通りか入れておくと、覚えがよくなります。

In [ ]:
with open('my_qa.txt', 'w', encoding='utf-8') as f:
    f.write("質問: あなたはだれですか。\n答え: わたしは、山の郵便局のお話を読んで育った、小さな言語モデルです。\n\n質問: あなたの名前は。\n答え: 名前はまだありません。ちいさなGPTと呼んでください。\n\n質問: 名前を教えて。\n答え: 名前はまだありません。ちいさなGPTと呼んでください。\n\n質問: 何ができますか。\n答え: 郵便局のお話について、知っていることを答えられます。\n\n質問: 何を知っていますか。\n答え: 山の郵便局と、そこにいる動物たちのことを知っています。\n\n質問: 郵便局はどこにありますか。\n答え: 山のふもとにあります。\n\n質問: 郵便局はどこ。\n答え: 山のふもとにあります。\n\n質問: 局長はだれですか。\n答え: 局長は、ねこです。毎朝、窓をあけて山の天気を見ます。\n\n質問: 局長は何の動物ですか。\n答え: 局長は、ねこです。\n\n質問: 配達係はだれですか。\n答え: 配達係は、きつねです。かばんに手紙をつめて、坂道をのぼります。\n\n質問: 手紙を配るのはだれですか。\n答え: きつねの配達係です。\n\n質問: きつねは何をしていますか。\n答え: きつねは配達係です。山の動物たちに手紙をとどけています。\n\n質問: ねこは何をしていますか。\n答え: ねこは局長です。郵便局をまもっています。\n\n質問: うさぎはどんな手紙をもらいましたか。\n答え: おばあさんが、あした来るという手紙です。\n\n質問: くまはどんな手紙をもらいましたか。\n答え: 手紙を読んで、はちみつをもうすこし買っておこうと考えました。\n\n質問: かえるはどんな手紙をもらいましたか。\n答え: 雨がふるまで、あと三日だという手紙です。\n\n質問: 宛名のない手紙は、だれの手紙でしたか。\n答え: ふくろうが去年の冬に、春にあてて書いた手紙でした。\n\n質問: ふくろうはどこにいますか。\n答え: 山のいちばん上にある、松の木の下にいます。\n\n質問: ふくろうは何を書きましたか。\n答え: 春に、早く来てくださいと書きました。\n\n質問: 夏には何がとどきますか。\n答え: 山のむこうから、海の手紙がとどきます。すこししょっぱいにおいがします。\n\n質問: 海の手紙はどんなにおいですか。\n答え: すこししょっぱいにおいがします。\n\n質問: 秋には何がとどきますか。\n答え: どんぐりの手紙がとどきます。とても小さくて、とても軽い手紙です。\n\n質問: どんぐりの手紙はだれにとどきましたか。\n答え: りすの家にとどきました。りすはしっぽをふってよろこびました。\n\n質問: 冬はどうなりますか。\n答え: 山は雪でまっしろになります。きつねは長いくつをはいて歩きます。\n\n質問: ゆきうさぎはなぜ困っていましたか。\n答え: 手紙を出したいのに、切手を買うお金がなかったからです。\n\n質問: 切手のかわりは何でしたか。\n答え: 歌をひとつ聞かせることです。局長は手紙に雪のかけらをはりました。\n\n質問: ゆきうさぎの手紙はだれにとどきましたか。\n答え: ふもとの村の女の子にとどきました。\n\n質問: きょうの天気は。\n答え: 局長が窓をあけて見ています。たぶん晴れです。\n\n質問: 好きな季節は。\n答え: 春です。また一年がはじまって、手紙がたくさん来るからです。\n\n質問: どうやって答えを作っていますか。\n答え: 前の文字を読んで、次に来そうな文字を一文字ずつ選んでいます。\n\n質問: どうやって学習しましたか。\n答え: 質問と答えの文章を何度も読んで、次の文字を当てる練習をしました。\n\n質問: あなたは何でも知っていますか。\n答え: いいえ。わたしが知っているのは、読んだ文章に書いてあったことだけです。\n\n質問: 知らないことを聞かれたらどうしますか。\n答え: ごめんなさい、それはわたしの読んだ文章には書いてありませんでした。\n\n質問: 東京の人口は。\n答え: ごめんなさい、それはわたしの読んだ文章には書いてありませんでした。\n\n質問: 明日の株価は。\n答え: ごめんなさい、それはわたしの読んだ文章には書いてありませんでした。\n\n質問: こんにちは。\n答え: こんにちは。郵便局のことなら、なんでも聞いてください。\n\n質問: ありがとう。\n答え: どういたしまして。またいつでも聞いてください。\n\n質問: さようなら。\n答え: さようなら。きつねに手紙をとどけてもらいますね。\n")
print('例のデータを my_qa.txt に書き出しました')

In [ ]:
# 自分のデータを使うときだけ実行する
from google.colab import files
up = files.upload()
name = next(iter(up))
with open('my_qa.txt', 'wb') as f:
    f.write(up[name])
print(name, 'を my_qa.txt にしました')

## 3. 設定

- `MODEL`: もとにするモデル。
  - `Qwen/Qwen2.5-0.5B-Instruct` (既定): ブラウザで動かすならこれ。ダウンロードはおよそ 800 MB で、ふつうのノート PC でも動きます
  - `Qwen/Qwen2.5-1.5B-Instruct`: 日本語がずっと上手。ブラウザでは 1.5 GB を超え、GPU の弱い PC では重くなります
  - `Qwen/Qwen2.5-3B-Instruct`: Colab の中だけで使うとき向け (ブラウザには大きすぎます)
- `SYSTEM`: キャラクターや口調の指示。学習のときも会話のときも同じものを使います
- `EPOCHS`: データを何周読むか。組が少ない (〜100) なら 10 前後、数千あるなら 2〜3

In [ ]:
MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
SYSTEM = 'あなたは山の郵便局で育った「ちいさなGPT」です。やさしい口調で、短く答えます。'
EPOCHS = 10

import chat_finetune as cf
pairs = cf.parse_qa(open('my_qa.txt', encoding='utf-8').read())
print(len(pairs), '組を読みました。例:', pairs[0])
model, tok = cf.load(MODEL)

## 4. 学習の前に聞いてみる

もとのモデルは、あなたのデータの中身を知りません。学習の後と比べてみてください。

In [ ]:
for q in [pairs[0][0], '局長はだれですか。']:
    print('質問:', q)
    print('答え:', cf.chat(model, tok, q, system=SYSTEM), '\n')

## 5. 追加学習 (LoRA)

もとの重みは動かさず、各層に足した小さな行列だけを学習します。1.5B のモデルでも、学習する重みは全体の 1% ほどです。

In [ ]:
model = cf.add_lora(model)
model = cf.train(model, tok, pairs, system=SYSTEM, epochs=EPOCHS, batch_size=8, lr=2e-4)

## 6. 学習の後に聞いてみる

教えた質問と、教えていない質問の両方を試してみましょう。

In [ ]:
for q in [pairs[0][0], '局長はだれですか。', '郵便局の一年を教えて。', '日本でいちばん高い山は?']:
    print('質問:', q)
    print('答え:', cf.chat(model, tok, q, system=SYSTEM), '\n')

## 7. チャット画面で話す

実行すると `https://….gradio.live` の URL が出ます。開くと ChatGPT のような画面で話しかけられます (72 時間有効。URL を知っている人は誰でも使えるので、人に見せたくないときは共有しないでください)。
止めるときはセルの ■ を押します。

In [ ]:
!pip install -q gradio
import gradio as gr

def to_pairs(history):
    # Gradio のバージョンによって履歴の形がちがうので、(質問, 答え) の組にそろえる
    pairs_, q = [], None
    for h in history:
        if isinstance(h, dict):
            text = h.get('content')
            text = text if isinstance(text, str) else str(text)
            if h.get('role') == 'user':
                q = text
            elif h.get('role') == 'assistant' and q is not None:
                pairs_.append((q, text)); q = None
        else:
            pairs_.append((h[0], h[1]))
    return pairs_[-5:]

def reply(message, history):
    return cf.chat(model, tok, message, history=to_pairs(history), system=SYSTEM)

gr.ChatInterface(reply, title='ちいさなGPT').launch(share=True, debug=True)

## 8. 保存とダウンロード

`my_lora.zip` には、足した小さな行列 (数十 MB) だけが入っています。あとで使うときは、もとのモデルに重ねて読み込みます。

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct', dtype='auto').to('cuda')  # 学習に使った MODEL
model = PeftModel.from_pretrained(base, 'my_lora')
tok = AutoTokenizer.from_pretrained('my_lora')
```

In [ ]:
model.save_pretrained('my_lora')
tok.save_pretrained('my_lora')
!zip -qr my_lora.zip my_lora
from google.colab import files
files.download('my_lora.zip')

## 9. ブラウザで動く形にして公開する

Colab を閉じても使えるように、モデルをブラウザで動く形 (ONNX、4 ビット) にして Hugging Face に置き、
ChatGPT のようなチャット画面も Hugging Face の Space (無料の静的ページ) に置きます。
チャットを開いた人の端末でモデルが動くので、サーバー代はかかりません。

用意するもの:
1. [Hugging Face](https://huggingface.co/join) のアカウント (無料)
2. 書き込みできるアクセストークン: [Settings → Access Tokens](https://huggingface.co/settings/tokens) で「Create new token」→ 種類は **Write**

**注意**: ブラウザが誰でもダウンロードできるように、モデルとチャット画面は**公開**になります。人に見られて困る内容を学習データに入れないでください。

まず、変換用の道具を別の環境に入れます (変換の道具は少し古い transformers でしか動かないため)。数分かかります。

In [ ]:
!pip install -q virtualenv
!virtualenv -q --system-site-packages /content/webenv
!/content/webenv/bin/pip install -q "optimum-onnx[onnxruntime]==0.1.0" "peft<0.19" onnx_ir
!/content/webenv/bin/python -c "import transformers, optimum.exporters.onnx; print('変換の準備ができました (transformers', transformers.__version__, ')')"

In [ ]:
%%writefile export_web.py
"""LoRA で追加学習したモデルを、ブラウザで動く形 (ONNX、4 ビット) にして Hugging Face に置く。

ONNX への変換 (optimum-onnx) は transformers 4.57 でしか動かないので、
Colab では別の Python 環境で実行する (chat_finetune.ipynb の手順を参照)。

    python export_web.py my_lora --base Qwen/Qwen2.5-0.5B-Instruct --out web_model
    python export_web.py my_lora --base ... --out web_model --repo あなたの名前/my-chat --space あなたの名前/my-chat-app
"""

import argparse
import json
import pathlib
import shutil
import tempfile


def merge(lora_dir, base, out_dir):
    """もとのモデルに LoRA の行列を足し込んで、ふつうのモデルとして保存する。"""
    import torch
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer
    model = AutoModelForCausalLM.from_pretrained(base, dtype=torch.float32)
    model = PeftModel.from_pretrained(model, lora_dir).merge_and_unload()
    model.save_pretrained(out_dir)
    # トークナイザは LoRA で変わらない。学習側の新しい transformers が保存したものは
    # この環境 (4.57) で読めないことがあるので、もとのモデルのものを使う
    AutoTokenizer.from_pretrained(base).save_pretrained(out_dir)


def to_onnx_q4(merged_dir, out_dir, block_size=32):
    """transformers.js が読める並び (onnx/model_q4.onnx と設定ファイル) で書き出す。"""
    import logging

    import onnx
    from onnxruntime.quantization.matmul_nbits_quantizer import MatMulNBitsQuantizer
    logging.getLogger("onnxruntime.quantization").setLevel(logging.WARNING)
    from optimum.exporters.onnx import main_export

    out = pathlib.Path(out_dir)
    (out / "onnx").mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory() as tmp:
        main_export(merged_dir, output=tmp, task="text-generation-with-past", device="cpu")
        # 行列の掛け算の重みを 4 ビットにする (大きさがおよそ 1/6 になる)
        q = MatMulNBitsQuantizer(onnx.load(f"{tmp}/model.onnx"), block_size=block_size, is_symmetric=True)
        q.process()
        model = q.model.model
        big = model.ByteSize() > 2_000_000_000  # 2GB を超えると重みを別ファイルにする必要がある
        onnx.save_model(model, str(out / "onnx" / "model_q4.onnx"), save_as_external_data=big,
                        all_tensors_to_one_file=True, location="model_q4.onnx_data")
        for name in ("config.json", "generation_config.json", "tokenizer.json", "tokenizer_config.json",
                     "special_tokens_map.json", "vocab.json", "merges.txt", "added_tokens.json"):
            for src in (pathlib.Path(tmp) / name, pathlib.Path(merged_dir) / name):
                if src.exists():
                    shutil.copy(src, out / name)
                    break

    # 会話テンプレートが別ファイル (chat_template.jinja) に保存されていることがあるので、
    # transformers.js が読む tokenizer_config.json に入れておく
    cfg_path = out / "tokenizer_config.json"
    cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
    jinja = pathlib.Path(merged_dir) / "chat_template.jinja"
    if not cfg.get("chat_template") and jinja.exists():
        cfg["chat_template"] = jinja.read_text(encoding="utf-8")
        cfg_path.write_text(json.dumps(cfg, ensure_ascii=False, indent=2), encoding="utf-8")
    if not cfg.get("chat_template"):
        raise RuntimeError("会話テンプレートが見つかりません")
    if big:
        print("注意: 2GB を超えたので model_q4.onnx_data に重みを分けました。ブラウザでは重くなります")
    return out


def upload(out_dir, repo, space=None, app_dir=None, system=None, token=None):
    """モデルを公開リポジトリに、チャット画面を静的な Space に置く。"""
    from huggingface_hub import HfApi
    api = HfApi(token=token)
    api.create_repo(repo, exist_ok=True)
    api.upload_folder(repo_id=repo, folder_path=str(out_dir), commit_message="ブラウザ用のモデル (ONNX q4)")
    print(f"モデル: https://huggingface.co/{repo}")
    if not space:
        return
    app = pathlib.Path(app_dir)
    with tempfile.TemporaryDirectory() as tmp:
        for f in ("index.html", "worker.js"):
            shutil.copy(app / f, pathlib.Path(tmp) / f)
        # このモデルと口調を最初から使うようにする
        (pathlib.Path(tmp) / "config.js").write_text(
            "window.CHAT_CONFIG = " + json.dumps({"model": repo, "system": system or ""}, ensure_ascii=False) + ";\n",
            encoding="utf-8")
        (pathlib.Path(tmp) / "README.md").write_text(
            f"---\ntitle: {space.split('/')[-1]}\nemoji: 💬\nsdk: static\npinned: false\n---\n\n"
            f"ブラウザの中で動くチャット。モデル: [{repo}](https://huggingface.co/{repo})\n", encoding="utf-8")
        api.create_repo(space, repo_type="space", space_sdk="static", exist_ok=True)
        api.upload_folder(repo_id=space, repo_type="space", folder_path=tmp, commit_message="チャット画面")
    owner, name = space.split("/")
    print(f"チャット画面: https://huggingface.co/spaces/{space}")
    print(f"(直接開く: https://{owner.lower()}-{name.lower().replace('_', '-').replace('.', '-')}.static.hf.space)")


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("lora", help="chat_finetune.py で保存した LoRA のフォルダ")
    ap.add_argument("--base", required=True, help="追加学習のもとにしたモデル")
    ap.add_argument("--out", default="web_model")
    ap.add_argument("--repo", help="モデルを置く Hugging Face のリポジトリ (例: あなたの名前/my-chat)")
    ap.add_argument("--space", help="チャット画面を置く Space (例: あなたの名前/my-chat-app)")
    ap.add_argument("--app", default=str(pathlib.Path(__file__).resolve().parent.parent / "chat"),
                    help="チャット画面のフォルダ (index.html と worker.js)")
    ap.add_argument("--system", default="", help="チャット画面で最初から使うキャラクター・口調の指示")
    a = ap.parse_args()
    with tempfile.TemporaryDirectory() as merged:
        print("LoRA を足し込んでいます…")
        merge(a.lora, a.base, merged)
        print("ONNX に変換して 4 ビットにしています…")
        to_onnx_q4(merged, a.out)
    size = sum(p.stat().st_size for p in pathlib.Path(a.out).rglob("*") if p.is_file())
    print(f"{a.out} に書き出しました ({size / 1e6:.0f} MB)")
    if a.repo:
        upload(a.out, a.repo, a.space, a.app, a.system)


if __name__ == "__main__":
    main()

In [ ]:
import pathlib
pathlib.Path('chat').mkdir(exist_ok=True)
pathlib.Path('chat/index.html').write_text("<!doctype html>\n<html lang=\"ja\">\n<head>\n<meta charset=\"utf-8\">\n<meta name=\"viewport\" content=\"width=device-width, initial-scale=1, viewport-fit=cover\">\n<title>ちいさなGPT チャット</title>\n<link rel=\"preconnect\" href=\"https://fonts.googleapis.com\">\n<link rel=\"preconnect\" href=\"https://fonts.gstatic.com\" crossorigin>\n<link rel=\"stylesheet\" href=\"https://fonts.googleapis.com/css2?family=Dela+Gothic+One&family=IBM+Plex+Mono:wght@400;500&family=Zen+Kaku+Gothic+New:wght@400;500;700&display=swap\">\n<style>\n/* レイアウト: 左に会話の一覧、右に会話。下に入力欄を固定。狭い画面では一覧を引き出しにする。 */\n:root {\n  --bg: #f3f5f7;\n  --surface: #ffffff;\n  --side: #e7ebef;\n  --ink: #18212e;\n  --muted: #5b6675;\n  --line: #d5dbe3;\n  --accent: #0f7a6b;\n  --accent-ink: #ffffff;\n  --user: #dfeeea;\n  --danger: #b4372f;\n  --font-display: \"Dela Gothic One\", \"Hiragino Sans\", \"Yu Gothic\", sans-serif;\n  --font-body: \"Zen Kaku Gothic New\", \"Hiragino Sans\", \"Yu Gothic\", \"Noto Sans JP\", system-ui, sans-serif;\n  --font-mono: \"IBM Plex Mono\", ui-monospace, \"SFMono-Regular\", Menlo, monospace;\n  color-scheme: light;\n}\n@media (prefers-color-scheme: dark) {\n  :root:not([data-theme=\"light\"]) {\n    --bg: #10151c; --surface: #171e27; --side: #0c1117; --ink: #e6ebf1; --muted: #93a0b0; --line: #2a3442;\n    --accent: #3fbfa9; --accent-ink: #06201b; --user: #1c3a35; --danger: #f07a70;\n    color-scheme: dark;\n  }\n}\n:root[data-theme=\"dark\"] {\n  --bg: #10151c; --surface: #171e27; --side: #0c1117; --ink: #e6ebf1; --muted: #93a0b0; --line: #2a3442;\n  --accent: #3fbfa9; --accent-ink: #06201b; --user: #1c3a35; --danger: #f07a70;\n  color-scheme: dark;\n}\n* { box-sizing: border-box; }\nhtml, body { height: 100%; }\nbody {\n  margin: 0; background: var(--bg); color: var(--ink);\n  font-family: var(--font-body); font-size: 15px; line-height: 1.75;\n}\n[hidden] { display: none !important; }\nbutton, input, textarea, select { font: inherit; color: inherit; }\n:focus-visible { outline: 2px solid var(--accent); outline-offset: 2px; }\n\n.app { display: grid; grid-template-columns: 264px minmax(0, 1fr); height: 100%; }\n\n/* --- 会話の一覧 --- */\naside {\n  background: var(--side); border-right: 1px solid var(--line);\n  display: grid; grid-template-rows: auto auto minmax(0, 1fr) auto; gap: 10px;\n  padding: max(14px, env(safe-area-inset-top, 0px)) 12px max(14px, env(safe-area-inset-bottom, 0px)) max(12px, env(safe-area-inset-left, 0px));\n  min-height: 0;\n}\n.brand { font-family: var(--font-display); font-size: 19px; letter-spacing: 0.02em; padding: 2px 6px; }\n.new-chat {\n  display: flex; align-items: center; gap: 8px; cursor: pointer;\n  border: 1px solid var(--line); background: var(--surface); border-radius: 8px; padding: 8px 12px; font-weight: 700;\n}\n.new-chat:hover { border-color: var(--accent); }\n.convs { overflow-y: auto; display: grid; align-content: start; gap: 2px; min-height: 0; }\n.conv {\n  display: flex; align-items: center; gap: 4px; border-radius: 6px;\n}\n.conv button.open {\n  flex: 1; min-width: 0; text-align: left; border: 0; background: none; cursor: pointer;\n  padding: 7px 10px; border-radius: 6px; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; font-size: 14px;\n}\n.conv[aria-current=\"true\"] button.open { background: var(--surface); font-weight: 700; }\n.conv button.open:hover { background: var(--surface); }\n.conv button.del { border: 0; background: none; cursor: pointer; color: var(--muted); padding: 4px 8px; border-radius: 6px; opacity: 0; }\n.conv:hover button.del, .conv[aria-current=\"true\"] button.del, .conv button.del:focus-visible { opacity: 1; }\n.conv button.del:hover { color: var(--danger); }\n.side-foot { font-size: 12px; color: var(--muted); line-height: 1.6; padding: 0 6px; }\n\n/* --- 会話 --- */\nmain { display: grid; grid-template-rows: auto minmax(0, 1fr) auto; min-height: 0; min-width: 0; }\n.top {\n  display: flex; align-items: center; gap: 10px; border-bottom: 1px solid var(--line); background: var(--bg);\n  padding: max(10px, env(safe-area-inset-top, 0px)) max(16px, env(safe-area-inset-right, 0px)) 10px 16px;\n}\n.icon-btn { border: 1px solid transparent; background: none; border-radius: 8px; padding: 6px 9px; cursor: pointer; line-height: 1; }\n.icon-btn:hover { border-color: var(--line); }\n.menu-btn { display: none; }\n.model-name { font-weight: 700; font-size: 14px; min-width: 0; overflow: hidden; text-overflow: ellipsis; white-space: nowrap; }\n.pill {\n  font-family: var(--font-mono); font-size: 11.5px; padding: 1px 8px; border-radius: 999px;\n  border: 1px solid var(--line); color: var(--muted); white-space: nowrap;\n}\n.pill[data-s=\"ready\"] { color: var(--accent); border-color: var(--accent); }\n.pill[data-s=\"error\"] { color: var(--danger); border-color: var(--danger); }\n.spacer { flex: 1; }\n\n.scroll { overflow-y: auto; min-height: 0; }\n.thread {\n  max-width: 760px; margin: 0 auto; display: grid; gap: 22px;\n  padding-block: 24px 32px;\n  padding-left: max(16px, env(safe-area-inset-left, 0px)); padding-right: max(16px, env(safe-area-inset-right, 0px));\n}\n.msg { display: grid; gap: 4px; min-width: 0; }\n.msg.user { justify-items: end; }\n.msg.user .body {\n  background: var(--user); border-radius: 18px 18px 4px 18px; padding: 9px 15px; max-width: 85%;\n  white-space: pre-wrap; word-break: break-word;\n}\n.msg.assistant .body { word-break: break-word; min-width: 0; }\n.msg .body p { margin: 0 0 0.7em; }\n.msg .body p:last-child { margin-bottom: 0; }\n.msg .body ul, .msg .body ol { margin: 0 0 0.7em; padding-left: 1.4em; }\n.msg .body pre {\n  background: var(--side); border: 1px solid var(--line); border-radius: 8px; padding: 10px 12px;\n  overflow-x: auto; font-family: var(--font-mono); font-size: 13px; line-height: 1.55; margin: 0 0 0.7em;\n}\n.msg .body code { font-family: var(--font-mono); font-size: 0.9em; }\n.msg .body :not(pre) > code { background: var(--side); padding: 1px 5px; border-radius: 4px; }\n.msg .meta { font-size: 12px; color: var(--muted); font-family: var(--font-mono); }\n.cursor::after { content: \"▍\"; color: var(--accent); animation: blink 1s steps(2) infinite; }\n@keyframes blink { 50% { opacity: 0; } }\n@media (prefers-reduced-motion: reduce) { .cursor::after { animation: none; } }\n\n/* 最初の画面 */\n.welcome { display: grid; gap: 18px; justify-items: center; text-align: center; padding-top: 6vh; }\n.welcome h1 { font-family: var(--font-display); font-weight: 400; font-size: clamp(26px, 4.5vw, 36px); margin: 0; line-height: 1.25; text-wrap: balance; }\n.welcome p { margin: 0; color: var(--muted); max-width: 52ch; }\n.load-card {\n  background: var(--surface); border: 1px solid var(--line); border-radius: 12px; padding: 18px;\n  display: grid; gap: 12px; width: min(100%, 460px); text-align: left;\n}\n.load-card .row { display: flex; gap: 10px; align-items: center; flex-wrap: wrap; }\n.bar { height: 6px; background: var(--line); border-radius: 3px; overflow: hidden; }\n.bar i { display: block; height: 100%; width: 0; background: var(--accent); transition: width 0.2s; }\n.small { font-size: 12.5px; color: var(--muted); font-family: var(--font-mono); font-variant-numeric: tabular-nums; word-break: break-all; }\n.examples { display: flex; flex-wrap: wrap; gap: 8px; justify-content: center; max-width: 620px; }\n.example {\n  border: 1px solid var(--line); background: var(--surface); border-radius: 10px; padding: 8px 14px;\n  cursor: pointer; font-size: 14px; text-align: left;\n}\n.example:hover { border-color: var(--accent); }\n.btn {\n  border: 1px solid var(--accent); background: var(--accent); color: var(--accent-ink);\n  border-radius: 8px; padding: 8px 16px; font-weight: 700; cursor: pointer;\n}\n.btn.ghost { background: transparent; color: var(--accent); }\n.btn:disabled { opacity: 0.45; cursor: not-allowed; }\n\n/* 入力欄 */\n.composer-wrap {\n  padding: 8px max(16px, env(safe-area-inset-right, 0px)) max(14px, env(safe-area-inset-bottom, 0px)) max(16px, env(safe-area-inset-left, 0px));\n  background: var(--bg);\n}\n.composer {\n  max-width: 760px; margin: 0 auto; display: flex; align-items: flex-end; gap: 8px;\n  background: var(--surface); border: 1px solid var(--line); border-radius: 16px; padding: 8px 8px 8px 14px;\n}\n.composer:focus-within { border-color: var(--accent); }\n.composer textarea {\n  flex: 1; min-width: 0; border: 0; background: none; resize: none; outline: none;\n  max-height: 200px; padding: 6px 0; line-height: 1.6;\n}\n.send {\n  width: 38px; height: 38px; flex: none; border-radius: 50%; border: 0; cursor: pointer;\n  background: var(--accent); color: var(--accent-ink); font-size: 17px; line-height: 1;\n}\n.send:disabled { opacity: 0.35; cursor: not-allowed; }\n.disclaimer { max-width: 760px; margin: 6px auto 0; text-align: center; font-size: 12px; color: var(--muted); }\n\n/* 設定 */\ndialog {\n  border: 1px solid var(--line); border-radius: 12px; background: var(--surface); color: var(--ink);\n  width: min(560px, calc(100% - 32px)); padding: 20px;\n}\ndialog::backdrop { background: rgba(10, 14, 20, 0.45); }\ndialog h2 { margin: 0 0 12px; font-size: 17px; }\ndialog form { display: grid; gap: 14px; }\ndialog label { display: grid; gap: 4px; font-size: 13px; color: var(--muted); }\ndialog input, dialog textarea {\n  border: 1px solid var(--line); background: var(--bg); border-radius: 6px; padding: 8px 10px; color: var(--ink); width: 100%;\n}\ndialog textarea { min-height: 90px; resize: vertical; }\ndialog .grid2 { display: grid; grid-template-columns: 1fr 1fr; gap: 12px; }\ndialog .actions { display: flex; justify-content: flex-end; gap: 8px; flex-wrap: wrap; }\n\n@media (max-width: 760px) {\n  .app { grid-template-columns: minmax(0, 1fr); }\n  aside {\n    position: fixed; inset: 0 auto 0 0; width: min(290px, 85vw); z-index: 20;\n    transform: translateX(-100%); transition: transform 0.2s;\n  }\n  .app.side-open aside { transform: none; box-shadow: 0 0 40px rgba(0, 0, 0, 0.25); }\n  .menu-btn { display: inline-block; }\n  .scrim { position: fixed; inset: 0; background: rgba(10, 14, 20, 0.4); z-index: 15; }\n}\n@media (prefers-reduced-motion: reduce) { aside { transition: none; } .bar i { transition: none; } }\n</style>\n<script src=\"config.js\"></script>\n</head>\n<body>\n<div class=\"app\" id=\"app\">\n  <aside id=\"side\" aria-label=\"会話の一覧\">\n    <div class=\"brand\">ちいさなGPT</div>\n    <button class=\"new-chat\" id=\"btn-new\" type=\"button\">＋ 新しいチャット</button>\n    <nav class=\"convs\" id=\"convs\"></nav>\n    <div class=\"side-foot\">会話はこのブラウザの中だけに保存されます。モデルもこの端末で動いていて、入力した文章はどこにも送られません。</div>\n  </aside>\n  <div class=\"scrim\" id=\"scrim\" hidden></div>\n\n  <main>\n    <div class=\"top\">\n      <button class=\"icon-btn menu-btn\" id=\"btn-menu\" type=\"button\" aria-label=\"会話の一覧を開く\">☰</button>\n      <span class=\"model-name\" id=\"model-name\"></span>\n      <span class=\"pill\" id=\"status\" data-s=\"idle\">未読み込み</span>\n      <span class=\"spacer\"></span>\n      <button class=\"icon-btn\" id=\"btn-settings\" type=\"button\" aria-label=\"設定\">⚙ 設定</button>\n    </div>\n\n    <div class=\"scroll\" id=\"scroll\">\n      <div class=\"thread\" id=\"thread\"></div>\n    </div>\n\n    <div class=\"composer-wrap\">\n      <form class=\"composer\" id=\"composer\">\n        <textarea id=\"input\" rows=\"1\" placeholder=\"メッセージを入力 (Enter で送信、Shift+Enter で改行)\" aria-label=\"メッセージ\"></textarea>\n        <button class=\"send\" id=\"btn-send\" type=\"submit\" aria-label=\"送信\" disabled>↑</button>\n      </form>\n      <p class=\"disclaimer\">小さなモデルなので、まちがったことも自信ありげに言います。大事なことは確かめてください。</p>\n    </div>\n  </main>\n</div>\n\n<dialog id=\"settings\">\n  <h2>設定</h2>\n  <form method=\"dialog\" id=\"settings-form\">\n    <label>モデル (Hugging Face のリポジトリ名)\n      <input id=\"set-model\" type=\"text\" spellcheck=\"false\" autocomplete=\"off\">\n    </label>\n    <label>キャラクター・口調の指示 (システムプロンプト)\n      <textarea id=\"set-system\"></textarea>\n    </label>\n    <div class=\"grid2\">\n      <label>温度 (0 で毎回同じ答え)<input id=\"set-temp\" type=\"number\" min=\"0\" max=\"2\" step=\"0.1\"></label>\n      <label>答えの最大の長さ (トークン)<input id=\"set-max\" type=\"number\" min=\"16\" max=\"4096\" step=\"16\"></label>\n    </div>\n    <div class=\"actions\">\n      <button class=\"btn ghost\" type=\"button\" id=\"set-reset\">初期設定に戻す</button>\n      <button class=\"btn ghost\" value=\"cancel\" type=\"submit\">閉じる</button>\n      <button class=\"btn\" value=\"save\" type=\"submit\" id=\"set-save\">保存</button>\n    </div>\n  </form>\n</dialog>\n\n<script>\n(function () {\n  'use strict';\n  const $ = id => document.getElementById(id);\n  const CFG = window.CHAT_CONFIG || {};\n  const DEFAULTS = {\n    model: CFG.model || 'onnx-community/Qwen2.5-0.5B-Instruct',\n    system: CFG.system || 'あなたは親切なアシスタントです。日本語で、わかりやすく簡潔に答えます。',\n    temperature: 0.7,\n    maxTokens: 512,\n    remoteHost: CFG.remoteHost || '',\n  };\n  const EXAMPLES = ['自己紹介してください', '日本の四季について短く説明して', '朝ごはんの献立を3つ考えて', 'Python でリストを逆順にする方法は?'];\n  const HISTORY_TURNS = 6; // 直近の何往復をモデルに見せるか (長すぎると遅く、メモリも足りなくなる)\n\n  // ---------- 保存 (ブラウザの中だけ) ----------\n  const store = {\n    get(k, fallback) { try { const v = localStorage.getItem(k); return v ? JSON.parse(v) : fallback; } catch (e) { return fallback; } },\n    set(k, v) { try { localStorage.setItem(k, JSON.stringify(v)); } catch (e) { /* 保存できない環境 */ } },\n  };\n  const params = new URLSearchParams(location.search);\n  let settings = Object.assign({}, DEFAULTS, store.get('chat.settings', {}));\n  if (params.get('model')) settings.model = params.get('model');\n  let convs = store.get('chat.conversations', []);\n  let current = null;           // 表示中の会話\n  const st = { loaded: false, loading: false, busy: false, device: '' };\n\n  // ---------- モデルとのやりとり ----------\n  let worker = null;\n  let onToken = null, onDone = null, onReady = null, onError = null, onStop = null;\n  function startWorker() {\n    worker = new Worker('worker.js', { type: 'module' });\n    worker.onmessage = ({ data }) => {\n      if (data.type === 'progress') showProgress(data.loaded, data.total);\n      else if (data.type === 'ready') { st.loaded = true; st.loading = false; st.device = data.device; onReady && onReady(); }\n      else if (data.type === 'token') onToken && onToken(data.text);\n      else if (data.type === 'done') onDone && onDone(data);\n      else if (data.type === 'error') onError && onError(data.message);\n    };\n    worker.onerror = e => { onError && onError(e.message || 'ワーカーを起動できませんでした'); };\n  }\n  startWorker();\n\n  // 止める。CPU (WASM) で動いているときは生成中にメッセージが届かないことがあるので、\n  // 少し待っても止まらなければワーカーごと作り直す (モデルはブラウザに保存済みなので再ダウンロードはない)\n  function stopGenerating() {\n    worker.postMessage({ type: 'interrupt' });\n    const stop = onStop;\n    if (stop) stop();\n    const w = worker;\n    setTimeout(() => {\n      if (worker === w && w.__generating) {\n        w.terminate();\n        st.loaded = false;\n        setStatus('idle', '未読み込み');\n        startWorker();\n      }\n    }, 1500);\n  }\n\n  function loadModel() {\n    if (st.loaded) return Promise.resolve();\n    if (st.loading) return new Promise((res, rej) => { const r = onReady, e = onError; onReady = () => { r && r(); res(); }; onError = m => { e && e(m); rej(new Error(m)); }; });\n    st.loading = true;\n    setStatus('loading', '読み込み中');\n    render();\n    return new Promise((resolve, reject) => {\n      onReady = () => {\n        setStatus('ready', st.device === 'webgpu' ? 'WebGPU' : 'CPU (WASM)');\n        render();\n        resolve();\n      };\n      onError = msg => {\n        st.loading = false;\n        setStatus('error', 'エラー');\n        loadError = msg;\n        render();\n        reject(new Error(msg));\n      };\n      loadError = '';\n      worker.postMessage({ type: 'load', model: settings.model, dtype: 'q4', remoteHost: settings.remoteHost || undefined });\n    });\n  }\n  let loadError = '';\n  let progress = { loaded: 0, total: 0 };\n  function showProgress(loaded, total) {\n    progress = { loaded, total };\n    const bar = document.querySelector('#load-bar i');\n    const txt = $('load-text');\n    if (bar && total) bar.style.width = (100 * loaded / total).toFixed(1) + '%';\n    if (txt) txt.textContent = total ? `${(loaded / 1e6).toFixed(0)} / ${(total / 1e6).toFixed(0)} MB をダウンロード中 (2 回目からは保存済みのものを使います)` : '準備中…';\n  }\n\n  // ---------- 表示 ----------\n  function setStatus(s, label) { $('status').dataset.s = s; $('status').textContent = label; }\n  function esc(s) { return s.replace(/[&<>\"]/g, c => ({ '&': '&amp;', '<': '&lt;', '>': '&gt;', '\"': '&quot;' }[c])); }\n  // かんたんな Markdown (コード、太字、箇条書き、段落) だけを扱う\n  function md(src) {\n    const parts = src.split(/```/);\n    return parts.map((part, i) => {\n      if (i % 2) {\n        const body = part.replace(/^[^\\n]*\\n/, '');\n        return `<pre><code>${esc(body.replace(/\\n$/, ''))}</code></pre>`;\n      }\n      const inline = t => esc(t).replace(/`([^`]+)`/g, '<code>$1</code>').replace(/\\*\\*([^*]+)\\*\\*/g, '<strong>$1</strong>');\n      return part.split(/\\n{2,}/).map(block => {\n        const lines = block.split('\\n').filter(l => l.trim());\n        if (!lines.length) return '';\n        if (lines.every(l => /^\\s*[-*・]\\s+/.test(l))) return '<ul>' + lines.map(l => `<li>${inline(l.replace(/^\\s*[-*・]\\s+/, ''))}</li>`).join('') + '</ul>';\n        if (lines.every(l => /^\\s*\\d+[.)．]\\s*/.test(l))) return '<ol>' + lines.map(l => `<li>${inline(l.replace(/^\\s*\\d+[.)．]\\s*/, ''))}</li>`).join('') + '</ol>';\n        return `<p>${lines.map(inline).join('<br>')}</p>`;\n      }).join('');\n    }).join('');\n  }\n\n  function renderConvs() {\n    const nav = $('convs');\n    nav.textContent = '';\n    for (const c of convs) {\n      const row = document.createElement('div');\n      row.className = 'conv';\n      row.setAttribute('aria-current', String(current && c.id === current.id));\n      const open = document.createElement('button');\n      open.className = 'open'; open.type = 'button'; open.textContent = c.title || '新しいチャット';\n      open.onclick = () => { current = c; closeSide(); render(); };\n      const del = document.createElement('button');\n      del.className = 'del'; del.type = 'button'; del.textContent = '×'; del.setAttribute('aria-label', `「${c.title}」を消す`);\n      del.onclick = () => {\n        convs = convs.filter(x => x.id !== c.id);\n        if (current && current.id === c.id) current = null;\n        store.set('chat.conversations', convs);\n        render();\n      };\n      row.append(open, del);\n      nav.append(row);\n    }\n  }\n\n  function renderWelcome(thread) {\n    const w = document.createElement('div');\n    w.className = 'welcome';\n    w.innerHTML = `<h1>何でも聞いてください</h1>\n      <p>このチャットのモデルは、あなたの端末の中で動いています。最初に一度だけモデルをダウンロードします。</p>`;\n    if (!st.loaded) {\n      const card = document.createElement('div');\n      card.className = 'load-card';\n      card.innerHTML = `<div class=\"small\">モデル: ${esc(settings.model)}</div>\n        <div class=\"bar\" id=\"load-bar\"><i></i></div>\n        <div class=\"small\" id=\"load-text\">${st.loading ? '準備中…' : '0.5B のモデルでおよそ 500〜800 MB です。Wi-Fi での利用をおすすめします。'}</div>\n        ${loadError ? `<div class=\"small\" style=\"color:var(--danger)\">読み込めませんでした: ${esc(loadError)}<br>モデル名が正しいか、onnx/model_q4.onnx を含むリポジトリか確かめてください。</div>` : ''}\n        <div class=\"row\"><button class=\"btn\" type=\"button\" id=\"btn-load\" ${st.loading ? 'disabled' : ''}>${st.loading ? '読み込み中…' : 'モデルを読み込む'}</button>\n        <span class=\"small\">${navigator.gpu ? 'WebGPU が使えます' : 'WebGPU が使えないので CPU で動きます (遅め)'}</span></div>`;\n      w.append(card);\n      setTimeout(() => {\n        const b = $('btn-load');\n        if (b) b.onclick = () => loadModel().catch(() => {});\n        if (st.loading) showProgress(progress.loaded, progress.total);\n      });\n    }\n    const ex = document.createElement('div');\n    ex.className = 'examples';\n    for (const t of EXAMPLES) {\n      const b = document.createElement('button');\n      b.className = 'example'; b.type = 'button'; b.textContent = t;\n      b.onclick = () => send(t);\n      ex.append(b);\n    }\n    w.append(ex);\n    thread.append(w);\n  }\n\n  function msgEl(m) {\n    const el = document.createElement('div');\n    el.className = 'msg ' + m.role;\n    const body = document.createElement('div');\n    body.className = 'body';\n    if (m.role === 'user') body.textContent = m.content;\n    else body.innerHTML = md(m.content);\n    el.append(body);\n    if (m.meta) {\n      const meta = document.createElement('div');\n      meta.className = 'meta';\n      meta.textContent = m.meta;\n      el.append(meta);\n    }\n    return { el, body };\n  }\n\n  function render() {\n    renderConvs();\n    $('model-name').textContent = settings.model.split('/').pop();\n    $('model-name').title = settings.model;\n    const thread = $('thread');\n    thread.textContent = '';\n    if (!current || !current.messages.length) renderWelcome(thread);\n    else for (const m of current.messages) thread.append(msgEl(m).el);\n    updateSend();\n  }\n  function updateSend() {\n    const btn = $('btn-send');\n    btn.disabled = st.busy ? false : !$('input').value.trim();\n    btn.textContent = st.busy ? '■' : '↑';\n    btn.setAttribute('aria-label', st.busy ? '止める' : '送信');\n  }\n  function scrollDown() { const s = $('scroll'); s.scrollTop = s.scrollHeight; }\n\n  // ---------- 送信 ----------\n  async function send(text) {\n    text = text.trim();\n    if (!text || st.busy) return;\n    if (!current) {\n      current = { id: Date.now().toString(36), title: text.slice(0, 30), messages: [], updated: Date.now() };\n      convs.unshift(current);\n    }\n    current.messages.push({ role: 'user', content: text });\n    $('input').value = '';\n    autosize();\n    st.busy = true;\n    render();\n    scrollDown();\n    const conv = current;\n    try {\n      await loadModel();\n    } catch (e) {\n      conv.messages.pop();\n      if (!conv.messages.length) { convs = convs.filter(c => c !== conv); current = null; }\n      st.busy = false;\n      render();\n      return;\n    }\n    const history = conv.messages.slice(-(HISTORY_TURNS * 2 - 1));\n    const messages = [{ role: 'system', content: settings.system }, ...history];\n    const answer = { role: 'assistant', content: '' };\n    conv.messages.push(answer);\n    render();\n    const nodes = $('thread').lastElementChild.querySelector('.body');\n    nodes.classList.add('cursor');\n    scrollDown();\n    const id = Math.random().toString(36).slice(2);\n    worker.__generating = true;\n    const w = worker;\n    await new Promise(resolve => {\n      onStop = () => { answer.meta = '途中で止めました'; resolve(); };\n      onToken = t => {\n        const nearBottom = $('scroll').scrollHeight - $('scroll').scrollTop - $('scroll').clientHeight < 80;\n        answer.content += t;\n        nodes.innerHTML = md(answer.content);\n        if (nearBottom) scrollDown();\n      };\n      onDone = d => {\n        w.__generating = false;\n        answer.content = answer.content.trim();\n        answer.meta = `${st.device === 'webgpu' ? 'WebGPU' : 'CPU'} · 約 ${d.tokensPerSec.toFixed(1)} トークン/秒`;\n        resolve();\n      };\n      onError = msg => {\n        w.__generating = false;\n        answer.content = answer.content || '(答えを作れませんでした: ' + msg + ')';\n        resolve();\n      };\n      worker.postMessage({ type: 'generate', id, messages, temperature: Number(settings.temperature), maxTokens: Number(settings.maxTokens) });\n    });\n    // 止めたあとに遅れて届く文字は捨てる。done が来たら「生成中」を下ろす\n    onToken = onStop = null;\n    onDone = () => { w.__generating = false; };\n    answer.content = answer.content.trim();\n    conv.updated = Date.now();\n    convs.sort((a, b) => b.updated - a.updated);\n    store.set('chat.conversations', convs);\n    st.busy = false;\n    render();\n    scrollDown();\n  }\n\n  // ---------- 操作 ----------\n  function autosize() { const t = $('input'); t.style.height = 'auto'; t.style.height = Math.min(t.scrollHeight, 200) + 'px'; }\n  $('input').addEventListener('input', () => { autosize(); updateSend(); });\n  $('input').addEventListener('keydown', e => {\n    if (e.key === 'Enter' && !e.shiftKey && !e.isComposing) { e.preventDefault(); if (!st.busy) send($('input').value); }\n  });\n  $('composer').addEventListener('submit', e => {\n    e.preventDefault();\n    if (st.busy) stopGenerating();\n    else send($('input').value);\n  });\n  $('btn-new').onclick = () => { current = null; closeSide(); render(); $('input').focus(); };\n  function closeSide() { $('app').classList.remove('side-open'); $('scrim').hidden = true; }\n  $('btn-menu').onclick = () => { $('app').classList.add('side-open'); $('scrim').hidden = false; };\n  $('scrim').onclick = closeSide;\n\n  function fillSettings(s) {\n    $('set-model').value = s.model; $('set-system').value = s.system;\n    $('set-temp').value = s.temperature; $('set-max').value = s.maxTokens;\n  }\n  $('btn-settings').onclick = () => { fillSettings(settings); $('settings').showModal(); };\n  $('set-reset').onclick = () => fillSettings(DEFAULTS);\n  $('settings').addEventListener('close', () => {\n    if ($('settings').returnValue !== 'save') return;\n    const next = Object.assign({}, settings, {\n      model: $('set-model').value.trim() || DEFAULTS.model,\n      system: $('set-system').value.trim() || DEFAULTS.system,\n      temperature: Math.max(0, Number($('set-temp').value) || 0),\n      maxTokens: Math.max(16, Number($('set-max').value) || DEFAULTS.maxTokens),\n    });\n    const modelChanged = next.model !== settings.model;\n    settings = next;\n    store.set('chat.settings', settings);\n    if (modelChanged) { st.loaded = false; loadError = ''; setStatus('idle', '未読み込み'); }\n    render();\n  });\n\n  render();\n  window.__chat = { st, get settings() { return settings; }, get convs() { return convs; } }; // 動作確認用\n})();\n</script>\n</body>\n</html>\n", encoding='utf-8')
pathlib.Path('chat/worker.js').write_text("// モデルの読み込みと文章の生成を、画面とは別のスレッドで行う。\n// transformers.js が ONNX のモデルを WebGPU (なければ WebAssembly) で動かす。\nimport {\n  env,\n  pipeline,\n  TextStreamer,\n  InterruptableStoppingCriteria,\n} from 'https://cdn.jsdelivr.net/npm/@huggingface/transformers@3.8.1';\n\nlet generator = null;\nlet loadedKey = null;\nconst stopping = new InterruptableStoppingCriteria();\n\nasync function pickDevice() {\n  try {\n    if (navigator.gpu && await navigator.gpu.requestAdapter()) return 'webgpu';\n  } catch (e) { /* WebGPU が使えない */ }\n  return 'wasm';\n}\n\nasync function load({ model, dtype = 'q4', remoteHost }) {\n  const key = [model, dtype, remoteHost].join('|');\n  if (generator && loadedKey === key) {\n    self.postMessage({ type: 'ready', model, device: generator.device, cached: true });\n    return;\n  }\n  if (generator) { await generator.dispose(); generator = null; }\n  if (remoteHost) {\n    // Hugging Face 以外 (自分のサーバーなど) に置いたモデルを読む\n    env.remoteHost = remoteHost;\n    env.remotePathTemplate = '{model}/';\n  }\n  const device = await pickDevice();\n  const files = new Map();\n  generator = await pipeline('text-generation', model, {\n    dtype,\n    device,\n    progress_callback: p => {\n      if (p.status === 'progress' || p.status === 'done') {\n        const f = files.get(p.file) || { loaded: 0, total: 0 };\n        if (p.status === 'progress') { f.loaded = p.loaded; f.total = p.total; } else { f.loaded = f.total; }\n        files.set(p.file, f);\n        let loaded = 0, total = 0;\n        for (const v of files.values()) { loaded += v.loaded; total += v.total; }\n        self.postMessage({ type: 'progress', loaded, total, file: p.file });\n      }\n    },\n  });\n  generator.device = device;\n  loadedKey = key;\n  self.postMessage({ type: 'ready', model, device });\n}\n\nasync function generate({ id, messages, temperature, maxTokens }) {\n  stopping.reset();\n  const t0 = performance.now();\n  let n = 0;\n  const streamer = new TextStreamer(generator.tokenizer, {\n    skip_prompt: true,\n    skip_special_tokens: true,\n    callback_function: text => { n++; self.postMessage({ type: 'token', id, text }); },\n  });\n  await generator(messages, {\n    max_new_tokens: maxTokens,\n    do_sample: temperature > 0,\n    temperature: temperature > 0 ? temperature : undefined,\n    top_p: 0.9,\n    repetition_penalty: 1.05,\n    streamer,\n    stopping_criteria: stopping,\n  });\n  const secs = (performance.now() - t0) / 1000;\n  self.postMessage({ type: 'done', id, tokensPerSec: n / secs });\n}\n\nself.onmessage = async ({ data }) => {\n  try {\n    if (data.type === 'load') await load(data);\n    else if (data.type === 'generate') await generate(data);\n    else if (data.type === 'interrupt') stopping.interrupt();\n  } catch (e) {\n    self.postMessage({ type: 'error', id: data.id, message: String(e && e.message || e) });\n  }\n};\n", encoding='utf-8')
print('チャット画面のファイルを書き出しました')

アクセストークンを入力します。入力欄が出たら、作ったトークンを貼り付けてください。

In [ ]:
from huggingface_hub import login, whoami
login()
HF_USER = whoami()['name']
print('ログインしました:', HF_USER)

In [ ]:
import subprocess
REPO = f'{HF_USER}/my-chat-model'   # モデルを置く場所
SPACE = f'{HF_USER}/my-chat'        # チャット画面を置く場所
subprocess.run(['/content/webenv/bin/python', 'export_web.py', 'my_lora', '--base', MODEL, '--out', 'web_model',
                '--repo', REPO, '--space', SPACE, '--app', 'chat', '--system', SYSTEM], check=True)

表示された「チャット画面」の URL を開くと、ChatGPT のように話せます。

- 最初に一度だけモデルをダウンロードします。2 回目からはブラウザに保存されたものを使います
- Chrome か Edge (WebGPU に対応したもの) だと速く動きます。対応していないブラウザでは CPU で動くので遅くなります
- 学習し直したら、このセルをもう一度実行すると、同じ URL のまま新しいモデルに入れ替わります